<a href="https://colab.research.google.com/github/ktrnandrea/Pengolahan-citra/blob/main/Aplikasi_OCR_Struk_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplikasi OCR Struk Belanja

Aplikasi ini berjalan di **Google Colab**. Unggah foto struk (JPG/PNG), lalu aplikasi akan:
1. Memperbaiki kualitas gambar secara sederhana (grayscale, denoise, threshold).
2. Membaca teks menggunakan Tesseract OCR (Bahasa Indonesia + Inggris).
3. Mendeteksi informasi umum seperti tanggal, waktu, dan total.
4. Menampilkan hasil OCR dan menyediakan unduhan hasil dalam format TXT.

> Catatan: hasil deteksi nominal/tanggal bersifat perkiraan. Periksa kembali hasilnya, terutama jika foto buram atau format struk tidak standar.


In [ ]:
# 1. Instalasi library dan OCR engine
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ind
!pip -q install gradio opencv-python-headless pytesseract pillow pandas


In [ ]:
# 2. Import library
import re
import cv2
import numpy as np
import pytesseract
import gradio as gr
from PIL import Image
from datetime import datetime
from IPython.display import display

print("Tesseract version:", pytesseract.get_tesseract_version())
print("Bahasa tersedia:", pytesseract.get_languages(config=''))


In [ ]:
# 3. Fungsi preprocessing gambar
def preprocess_receipt(pil_image):
    # Konversi PIL ke OpenCV
    image = cv2.cvtColor(np.array(pil_image.convert("RGB")), cv2.COLOR_RGB2BGR)

    # Perbesar gambar agar teks kecil lebih mudah dibaca
    height, width = image.shape[:2]
    if width < 1600:
        scale = 1600 / width
        image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)

    # Adaptive threshold membantu untuk struk dengan pencahayaan tidak merata
    processed = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 11
    )
    return processed


In [ ]:
# 4. Fungsi OCR dan ekstraksi informasi struk
def extract_receipt_fields(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    # Cari tanggal dengan beberapa pola umum
    date_patterns = [
        r'\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b',
        r'\b(\d{4}[/-]\d{1,2}[/-]\d{1,2})\b'
    ]
    date_value = "-"
    for pattern in date_patterns:
        match = re.search(pattern, text)
        if match:
            date_value = match.group(1)
            break

    # Cari waktu, misalnya 14:35 atau 14.35
    time_match = re.search(r'\b([01]?\d|2[0-3])[:.]([0-5]\d)\b', text)
    time_value = time_match.group(0).replace('.', ':') if time_match else "-"

    # Cari baris yang mengandung kata total; ambil nominal pada baris tersebut
    total_value = "-"
    total_keywords = r'total|grand\s*total|jumlah\s*bayar|total\s*bayar|tagihan'
    for line in lines:
        if re.search(total_keywords, line, re.IGNORECASE):
            amounts = re.findall(r'(?:Rp\s*)?([0-9][0-9.,]*)', line, re.IGNORECASE)
            if amounts:
                total_value = amounts[-1]
                break

    # Jika tidak ditemukan lewat kata kunci, cari nilai terbesar sebagai petunjuk
    if total_value == "-":
        candidates = re.findall(r'(?<!\w)(?:Rp\s*)?([0-9][0-9.,]{2,})(?!\w)', text, re.IGNORECASE)
        if candidates:
            def numeric_value(value):
                digits = re.sub(r'\D', '', value)
                return int(digits) if digits else 0
            total_value = max(candidates, key=numeric_value)

    # Nama toko diperkirakan dari baris pertama
    merchant = lines[0][:80] if lines else "-"

    return {
        "Nama toko (perkiraan)": merchant,
        "Tanggal (perkiraan)": date_value,
        "Waktu (perkiraan)": time_value,
        "Total (perkiraan)": total_value
    }


def run_receipt_ocr(image):
    if image is None:
        return "Silakan unggah foto struk terlebih dahulu.", {}, None

    processed = preprocess_receipt(image)

    # PSM 6: menganggap gambar berisi satu blok teks
    config = "--oem 3 --psm 6"
    ocr_text = pytesseract.image_to_string(
        processed, lang="ind+eng", config=config
    )

    fields = extract_receipt_fields(ocr_text)
    preview = Image.fromarray(processed)

    summary = "\n".join([f"**{key}:** {value}" for key, value in fields.items()])
    return ocr_text.strip() or "Teks tidak berhasil dibaca. Coba foto yang lebih jelas.", fields, preview


In [ ]:
# 5. Antarmuka aplikasi menggunakan Gradio
with gr.Blocks(title="OCR Struk Belanja") as demo:
    gr.Markdown("# 🧾 OCR Struk Belanja")
    gr.Markdown("Unggah foto struk untuk membaca teks dan memperkirakan informasi transaksi.")

    with gr.Row():
        input_image = gr.Image(type="pil", label="Unggah foto struk")
        processed_image = gr.Image(type="pil", label="Hasil preprocessing")

    run_button = gr.Button("Proses Struk", variant="primary")

    with gr.Tab("Informasi Transaksi"):
        fields_output = gr.JSON(label="Hasil ekstraksi (perkiraan)")

    with gr.Tab("Teks OCR"):
        text_output = gr.Textbox(label="Teks hasil OCR", lines=18)

    run_button.click(
        fn=run_receipt_ocr,
        inputs=input_image,
        outputs=[text_output, fields_output, processed_image]
    )

demo.launch(share=True, debug=True)


In [ ]:
print("hello world")

## Cara menjalankan
1. Buka [Google Colab](https://colab.research.google.com/).
2. Pilih **File → Upload notebook**, lalu unggah file notebook ini.
3. Jalankan setiap cell dari atas ke bawah.
4. Pada cell terakhir, buka tautan Gradio yang muncul (biasanya berakhiran `gradio.live`).
5. Unggah foto struk dan tekan **Proses Struk**.

## Tips agar OCR lebih akurat
- Foto struk dari atas, tidak miring, dan seluruh bagian struk terlihat.
- Pastikan tulisan tidak blur dan pencahayaan merata.
- Hindari bayangan atau pantulan cahaya.
- Jika hasil total keliru, cek kembali teks OCR karena format nominal tiap toko bisa berbeda.
